# Ejemplo de regresión Heckman - Modelo de selección de muestra

## 1. Importación de bibliotecas y datos

In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.iolib.summary2 import summary_col
import heckman as heckman

# Omiting WARNINGS
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Read data
df = pd.read_stata('womenwk.dta')
df

El modelo de Heckman distingue dos ecuaciones. La primera es una **ecuación de selección**: el salario sólo se observa para las mujeres que trabajan, y la decisión de trabajar se modela con un probit sobre una variable latente $s_i^*$:
$$s_i^* = \gamma_0 + \gamma_1 married_i + \gamma_2 children_i + \gamma_3 educ_i + \gamma_4 age_i + v_i, \qquad s_i = 1 \text{ si } s_i^* > 0$$

La segunda es la **ecuación de interés**, el salario, que sólo se observa cuando $s_i = 1$:
$$wage_i = \beta_0 + \beta_1 educ_i + \beta_2 age_i + u_i$$

Los errores $u_i$ y $v_i$ tienen correlación $\rho$. En resumen, el salario por hora es función de la educación y la edad, mientras que la probabilidad de trabajar (la probabilidad de que se observe el salario) depende además del estado civil y del número de hijos. Esas dos variables son la **restricción de exclusión**: afectan la decisión de trabajar, pero se supone que no afectan el salario ofrecido.

## 2. Generación y transformación de datos

In [ ]:
#Variables
Y = df.wage
W = df[['married', 'children', 'education', 'age']]
W = sm.add_constant(W)

X = df[['education', 'age']]
X = sm.add_constant(X)

## 3. Estimación

In [ ]:
# En dos etapas
res = heckman.Heckman(Y, X, W).fit(method = 'twostep')
print(res.summary())

> **Cuidado con el orden de los argumentos.** La firma es `Heckman(endog, exog, exog_select)`: primero la variable dependiente, después los regresores de la **ecuación de salario** (`X`) y al final los de la **ecuación de selección** (`W`). Si se intercambian `X` y `W`, el código corre sin ningún aviso, pero estima otro modelo: `married` y `children` entran en la ecuación de salario, la restricción de exclusión queda al revés, y la correlación estimada sale $\hat\rho = 1.23$, fuera del intervalo $[-1, 1]$.
>
> Con el orden correcto, la estimación reproduce el ejemplo de este mismo archivo en el manual de Stata (`heckman wage educ age, select(married children educ age) twostep`): `education` = 0.9825, `age` = 0.2119, constante = 0.7340, $\hat\lambda = 4.0016$, $\hat\rho = 0.6728$ y $\hat\sigma = 5.9474$.

In [ ]:
#
Y = df[df.wage > 0].wage
X = df[df.wage > 0][['education', 'age']]
X = sm.add_constant(X)

In [ ]:
est = sm.OLS(Y, X).fit()
print(est.summary())

### Comparación: MCO sobre la muestra seleccionada frente a Heckman

MCO sobre las 1,343 mujeres que trabajan da un rendimiento de la educación de 0.897; la corrección de Heckman lo lleva a 0.983, y el de la edad pasa de 0.147 a 0.212. El sentido del sesgo es el que anticipa el cap. 8 de las notas: con $\rho > 0$, las mujeres con poca educación que aun así trabajan tienen, en promedio, no observables altos, lo que aplana la pendiente estimada por MCO.